# **8. Эксперименты с наборами признаков (Feature Set Experiments): оценка вклада групп признаков**

* __Цель:__ определить влияние временных, календарных, погодных, lag- и rolling-признаков на качество прогнозирования транспортной нагрузки.
* __Задачи:__
  - сравнить пять feature scenarios на одном validation-периоде;
  - использовать CatBoost как основную модель анализа;
  - проверить устойчивость выводов с помощью XGBoost и Random Forest;
  - сопоставить сценарии по MAE, RMSE, MAPE и R².
* __Алгоритм выполнения:__
  1. Загрузить scenario registry и generated feature set experiment outputs.
  2. Проверить единообразие validation sample для всех экспериментов.
  3. Проанализировать результаты основной модели CatBoost.
  4. Выполнить robustness-check по XGBoost и Random Forest.
  5. Построить validation RMSE comparison plot.
  6. Выполнить methodological audit.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

from traffic_forecasting.config import (
    FEATURE_SET_COMPARISON_FIGURE_PATH,
    FEATURE_SET_COMPARISON_PATH,
    FEATURE_SET_METRICS_PATH,
    FEATURE_SET_PREDICTIONS_PATH,
    PROCESSED_DATA_PATH,
)
from traffic_forecasting.feature_sets import (
    get_feature_set_columns,
    get_feature_set_scenarios,
)
from traffic_forecasting.visualization import plot_feature_set_comparison

FEATURE_SET_INPUTS = (
    PROCESSED_DATA_PATH,
    FEATURE_SET_METRICS_PATH,
    FEATURE_SET_COMPARISON_PATH,
    FEATURE_SET_PREDICTIONS_PATH,
)
missing_inputs = [path for path in FEATURE_SET_INPUTS if not path.is_file()]
if missing_inputs:
    missing_text = ", ".join(str(path) for path in missing_inputs)
    raise FileNotFoundError(
        f"Missing feature set inputs: {missing_text}. "
        "Run scripts/run_pipeline.py and scripts/run_experiments.py --feature-sets first."
    )

## **8.1. Определение сценариев признакового пространства (Feature Scenario Definitions)**

In [ ]:
scenarios = get_feature_set_scenarios()
scenario_summary = pd.DataFrame(
    [
        {
            "feature_set": scenario.name,
            "label": scenario.label,
            "feature_groups": ", ".join(scenario.feature_groups),
            "feature_count": len(get_feature_set_columns(scenario.name)),
        }
        for scenario in scenarios.values()
    ]
)

display(Markdown("### **Feature set scenario registry**"))
display(scenario_summary)

## **8.2. Загрузка результатов эксперимента (Loading Experiment Results)**

In [ ]:
metrics = pd.read_csv(FEATURE_SET_METRICS_PATH)
comparison = pd.read_csv(FEATURE_SET_COMPARISON_PATH)
predictions = pd.read_csv(
    FEATURE_SET_PREDICTIONS_PATH,
    parse_dates=["date_time"],
)

output_summary = pd.DataFrame(
    {
        "output": ["metrics", "comparison", "validation_predictions"],
        "rows": [len(metrics), len(comparison), len(predictions)],
        "columns": [metrics.shape[1], comparison.shape[1], predictions.shape[1]],
    }
)

display(Markdown("### **Feature set output summary**"))
display(output_summary)
display(Markdown("### **Validation metrics for all feature scenarios**"))
display(metrics)

## **8.3. Проверка общего validation sample (Shared Validation Sample Audit)**

In [ ]:
sample_audit = predictions.groupby(["feature_set", "model", "split"], as_index=False).agg(
    rows=("date_time", "size"),
    validation_start=("date_time", "min"),
    validation_end=("date_time", "max"),
    actual_target_sum=("actual_traffic_volume", "sum"),
)

display(Markdown("### **Shared validation sample audit**"))
display(sample_audit)

assert set(sample_audit["split"]) == {"validation"}
assert sample_audit["rows"].nunique() == 1
assert sample_audit["validation_start"].nunique() == 1
assert sample_audit["validation_end"].nunique() == 1
assert sample_audit["actual_target_sum"].nunique() == 1

## **8.4. Основной анализ по CatBoostRegressor (Primary CatBoost Analysis)**

In [ ]:
catboost_results = comparison.query("model == 'catboost'").sort_values("rank_within_model")

display(Markdown("### **CatBoost feature set ranking by validation RMSE**"))
display(
    catboost_results[
        [
            "rank_within_model",
            "feature_set_label",
            "feature_count",
            "mae",
            "rmse",
            "mape",
            "r2",
            "rmse_improvement_vs_temporal_calendar_pct",
        ]
    ]
)

## **8.5. Проверка устойчивости выводов (Robustness Check Across Models)**

In [ ]:
best_by_model = comparison.query("rank_within_model == 1").sort_values("rmse")
lag_scenarios = comparison[
    comparison["feature_set"].isin(
        ["temporal_calendar_lag", "temporal_calendar_lag_rolling", "full"]
    )
]

display(Markdown("### **Best feature scenario for each model**"))
display(
    best_by_model[
        [
            "model",
            "feature_set_label",
            "rmse",
            "rmse_improvement_vs_temporal_calendar_pct",
        ]
    ]
)

display(Markdown("### **Lag-containing scenarios across robustness models**"))
display(
    lag_scenarios[["model", "feature_set_label", "rmse", "rank_within_model"]].sort_values(
        ["model", "rank_within_model"]
    )
)

## **8.6. Визуальное сравнение feature scenarios (Feature Set Comparison Plot)**

In [ ]:
display(Markdown("### **Validation RMSE across feature scenarios and models**"))
figure, _ = plot_feature_set_comparison(
    comparison,
    metric="rmse",
    output_path=FEATURE_SET_COMPARISON_FIGURE_PATH,
)
plt.show()

## **8.7. Аудит методологических ограничений (Methodological Constraints Audit)**

In [ ]:
methodological_audit = pd.DataFrame(
    {
        "check": [
            "Metrics contain validation split only",
            "Predictions contain validation split only",
            "All scenarios use the same validation rows",
            "Selected models match feature set experiment design",
            "Target remains traffic_volume",
            "Feature set comparison figure was generated",
        ],
        "passed": [
            set(metrics["split"]) == {"validation"},
            set(predictions["split"]) == {"validation"},
            sample_audit["rows"].nunique() == 1,
            set(metrics["model"]) == {"catboost", "xgboost", "random_forest"},
            "actual_traffic_volume" in predictions.columns,
            FEATURE_SET_COMPARISON_FIGURE_PATH.is_file(),
        ],
    }
)

display(Markdown("### **Methodological audit results**"))
display(methodological_audit)

assert methodological_audit["passed"].all()

## **8.8. Анализ и интерпретация результатов экспериментов (Analysis and Interpretation of Feature Set Experiments)**

В рамках эксперимента по исследованию состава признаков была выполнена оценка влияния различных групп входных переменных на качество прогнозирования транспортной нагрузки. Данный эксперимент продолжает предшествующие этапы построения baseline- и ensemble-моделей и направлен на определение того, какие группы признаков вносят наибольший вклад в точность прогноза.

**Ключевые результаты:**
1. **Сформированы несколько сценариев состава признаков.**
   Для оценки вклада отдельных групп признаков были рассмотрены пять сценариев: временные и календарные признаки, временные + календарные + погодные признаки, временные + календарные + лаговые признаки, временные + календарные + лаговые + rolling-признаки, а также полный набор признаков.
2. **Сравнение выполнялось на единой validation-выборке.**
   Все feature set scenarios оценивались на одном и том же validation-периоде. Это важно для корректности эксперимента, поскольку различия в метриках должны быть обусловлены именно составом признаков, а не разными временными интервалами или различным числом наблюдений. В качестве основных метрик использовались `MAE`, `RMSE`, `MAPE` и `R²`, при этом основным критерием сравнения выступала `RMSE`.
3. **Preprocessing строился отдельно для каждого сценария признаков.**
   Для каждого feature scenario использовался собственный preprocessing pipeline, построенный только на выбранных признаках. Это позволяет корректно обрабатывать reduced feature sets, например сценарии без погодных категориальных признаков `weather_main` и `weather_description`.
4. **Основной прирост качества обеспечили лаговые признаки.**
   Наиболее заметное снижение ошибки наблюдается при добавлении lag-признаков, отражающих исторические значения транспортной нагрузки. Для модели `CatBoostRegressor` лучший результат был получен на сценарии `Temporal + calendar + lag`, где используются временные, календарные и лаговые признаки. Это показывает, что предыдущие значения транспортного потока несут ключевую информацию для краткосрочного прогнозирования нагрузки автотранспортной системы.
5. **Погодные признаки сами по себе дали ограниченный прирост качества.**
   Сценарий `Temporal + calendar + weather` показал улучшение относительно базового набора временных и календарных признаков, однако это улучшение оказалось существенно меньше, чем эффект от добавления лаговых признаков. Следовательно, погодные параметры могут быть полезны как дополнительный источник информации, но в рассматриваемой задаче они не являются главным фактором повышения точности прогноза.
6. **Rolling-признаки не дали значимого улучшения относительно lag-only сценария.**
   Добавление rolling-признаков к временным, календарным и лаговым переменным не привело к заметному улучшению результата для основной модели. Это может объясняться тем, что часть информации, содержащейся в rolling statistics, уже частично представлена в lag-признаках. Таким образом, rolling-признаки могут повышать интерпретируемость динамики временного ряда, но не обязательно улучшают качество прогноза в условиях уже сильного набора lag-признаков.
7. **Полный набор признаков оказался близок к лучшему результату, но не превзошел его для основной модели.**
   Полный feature set, включающий временные, календарные, погодные, lag- и rolling-признаки, показал качество, близкое к лучшему результату, однако для `CatBoostRegressor` он не превзошел сценарий `Temporal + calendar + lag`. Это указывает на то, что расширение признакового пространства не всегда приводит к улучшению качества модели. В некоторых случаях дополнительные признаки могут быть избыточными или давать небольшой вклад по сравнению с историческими значениями целевой переменной.
8. **Robustness-check подтвердил важность lag-признаков.**
   Для проверки устойчивости вывода дополнительно использовались модели `XGBRegressor` и `RandomForestRegressor`. Несмотря на различия в ранжировании отдельных сценариев, сильные результаты во всех случаях связаны с наличием lag-признаков. Это подтверждает, что исторические значения транспортной нагрузки являются наиболее информативной группой признаков для рассматриваемой задачи прогнозирования.

**Итоговое методологическое резюме:** эксперимент по исследованию состава признаков показал, что наибольший вклад в качество краткосрочного прогнозирования транспортной нагрузки вносят лаговые признаки, отражающие историческую динамику транспортного потока. Временные и календарные признаки формируют базовый уровень качества, погодные признаки дают ограниченный дополнительный эффект, а rolling-признаки не обеспечили существенного улучшения относительно lag-only сценария.